In [30]:
target_path = '/home/gina101/new_data/Eleusis 2.0/Environment/Session1/'

In [4]:
import fnmatch
import os
import pandas as pd
import shutil
import itertools

In [6]:
# source_path = r'/content/drive/MyDrive/TT8/Sociodrama 6/MAX - superepisodes'
source_path = '/home/gina101/Eleusis 2.0/Eleusis 2.0/'
ids_path = os.path.join(source_path, 'IDs')
scenes_path = os.path.join(source_path, 'Scenes')
tasks_path = os.path.join(source_path, 'Tasks')


# Small sanitization script for Eleusis 2.0 data

In [11]:
ids = os.listdir(ids_path)
scenes = [15, 18, 25, 36, 41, 50, 54, 76, 81, 117, 119, 123, 161]
# Find all files that match this scene
for scene in scenes:
    scene_files = fnmatch.filter(ids, "idScene{}*".format(scene))
    # Read each file and compare the values of their first cell and their second one
    if len(scene_files) > 1:
        print("Scene ", scene, " files: ", scene_files)
        for file in scene_files:
            df = pd.read_excel(os.path.join(ids_path, file), header=None)
            print("File: ", file, " First cell: ", df.iloc[0,0], " Second cell: ", df.iloc[1,0])

Scene  25  files:  ['idScene25SCSensor5-Introduction-5.xlsx', 'idScene25HRSensor3-Introduction-5.xlsx']
File:  idScene25SCSensor5-Introduction-5.xlsx  First cell:  22380  Second cell:  22763
File:  idScene25HRSensor3-Introduction-5.xlsx  First cell:  22370  Second cell:  22753
Scene  50  files:  ['idScene50SCSensor4-Main-2.xlsx', 'idScene50TempSensor4-Main-2.xlsx', 'idScene50TempSensor6-Main-2.xlsx', 'idScene50SCSensor5-Main-2.xlsx', 'idScene50HRSensor5-Main-2.xlsx', 'idScene50HRSensor4-Main-2.xlsx', 'idScene50TempSensor2-Main-2.xlsx']
File:  idScene50SCSensor4-Main-2.xlsx  First cell:  60134  Second cell:  90444
File:  idScene50TempSensor4-Main-2.xlsx  First cell:  60134  Second cell:  90444
File:  idScene50TempSensor6-Main-2.xlsx  First cell:  60124  Second cell:  90437
File:  idScene50SCSensor5-Main-2.xlsx  First cell:  60105  Second cell:  90402
File:  idScene50HRSensor5-Main-2.xlsx  First cell:  60105  Second cell:  90402
File:  idScene50HRSensor4-Main-2.xlsx  First cell:  60134  

# Extract Scene Data
1. Find ID file for each scene
2. Find line that corresponds to each timestamp
3. Find all task files that this scene belongs to
4. 'Splice' and only keep these files
5. Join them together in one dataframe
6. Write them to a file

## Modification for Ljubljana (?)

In [4]:
def findSensorNoInTaskFiles(name):
  # denominators = ["HR", "SC", "Temp"]
  # for denom in denominators:
  #   if denom in name:
  #     parts = name.split(denom)
  #     if len(parts) > 1:
  #         # return parts[1].split('-')[0]
  #         return parts[1].split('.xlsx')[0]
  return name.split(".xlsx")[0].split("Sensor")[1]
  # return None

# def sceneNo(name):
#   denominators = ["HR", "SC", "Temp"]
#   for denom in denominators:
#     if denom in name:
#       parts = name.split(denom)
#       if len(parts) > 1:
#         return parts[0].split("Scene")[1]

def sceneNo(name):
  return name.split("_")[0].split("Scene")[1]

In [ ]:
def findTaskNo(name):
  return name.split("Task")[1].split('.xlsx')[0]

: 

In [ ]:
# Find Scene ID files
id_files = os.listdir(ids_path)
scene_files = os.listdir(scenes_path)
task_files = os.listdir(tasks_path)

# Print these files
print("All idScene files found: ", id_files)
print("All task files found: ", task_files)

processed_scenes = []

def get_sensor_type(filename):
    if "HR" in filename: return "HR"
    if "SC" in filename: return "SC"
    if "Temp" in filename: return "Temp"
    return None

for id_scene_file in id_files:
  all_filtered_task_files = []
  # Find Scene No and Task No from the id_scene_file
  scene_num = sceneNo(id_scene_file)
  if scene_num in processed_scenes:
    continue
  processed_scenes.append(scene_num)

  task_num = findTaskNo(id_scene_file)
  print(f"Processing idScene file: {id_scene_file} (Scene no: {scene_num}, Task no: {task_num})")

  task_files_for_this_scene = fnmatch.filter(task_files, "Task{}*".format(task_num))
  # print(f"Task files for this scene: {task_files_for_this_scene}")
  assert(len(task_files_for_this_scene) == 12)

  # Open id_scene_file and read cells A1 and A2
  id_scene_file_path = os.path.join(ids_path, id_scene_file)
  df_id_scene = pd.read_excel(id_scene_file_path, header=None)
  # print(df_id_scene)
  starting_timestamp = df_id_scene.iloc[0, 0].astype(int)
  ending_timestamp = df_id_scene.iloc[1, 0].astype(int)

  # # Now, time to find the appropriate lines to 'splice' the files
  # # Read through each one of the task files and find the lines that have the values of cell_a1 and cell_a2

  for task_file in task_files_for_this_scene:
      task_file_path = os.path.join(tasks_path, task_file)
      df_task = pd.read_excel(task_file_path)

      # Convert the first column of df_task to integer type for comparison
      start_index_series = df_task[df_task.iloc[:, 0].astype(int) == starting_timestamp].index
      if start_index_series.empty:
        print("Start index empty")
      end_index_series = df_task[df_task.iloc[:, 0].astype(int) == ending_timestamp].index
      if end_index_series.empty:
        print("End index empty")

      if not start_index_series.empty and not end_index_series.empty:
          start_row = start_index_series[0]
          end_row = end_index_series[0]
          if start_row <= end_row:
            # Create new dataframe and copy the rows
            spliced_df = df_task.iloc[start_row : end_row + 1]

            # Extract sensor type and number for column naming
            sensor_type = get_sensor_type(task_file)
            sensor_num = findSensorNoInTaskFiles(task_file)
            print(f"Task file {task_file} has sensor type {sensor_type} and belongs to participant {sensor_num}")

            if sensor_type and sensor_num:
              # Assuming the first column is 'Timestamp' and the second is the value
              spliced_df.columns = ['Timestamp', 'Value']

          else:
            print(f"Warning: Start index ({start_row}) is greater than end index ({end_row}) for {task_file}. Skipping splicing.")

          output_filename = f"Scene{scene_num}_{sensor_type}Sensor{sensor_num}_Task{task_num}.xlsx"
          output_path = os.path.join(target_path, output_filename)
          spliced_df.to_excel(output_path, index=False)
          print(f"Saved spliced data for {task_file} to {output_path}")
      else:
        print(f"No data merged for Scene {scene_num}, Task {task_num}")

All idScene files found:  ['idScene6_TempSensor3_Task6.xlsx', 'idScene2_SCSensor4_Task2.xlsx', 'idScene4_SCSensor4_Task5.xlsx', 'idScene2_HRSensor2_Task2.xlsx', 'idScene1_SCSensor4_Task1.xlsx', 'idScene1_SCSensor1_Task1.xlsx', 'idScene3_SCSensor3_Task4.xlsx', 'idScene6_SCSensor4_Task6.xlsx', 'idScene1_TempSensor2_Task1.xlsx', 'idScene3_HRSensor2_Task4.xlsx', 'idScene1_SCSensor3_Task1.xlsx', 'idScene5_TempSensor4_Task6.xlsx', 'idScene2_TempSensor1_Task2.xlsx', 'idScene2_TempSensor4_Task2.xlsx', 'idScene2_SCSensor3_Task2.xlsx', 'idScene1_TempSensor3_Task1.xlsx', 'idScene5_TempSensor2_Task6.xlsx', 'idScene3_TempSensor4_Task4.xlsx', 'idScene2_SCSensor2_Task2.xlsx', 'idScene4_HRSensor2_Task5.xlsx', 'idScene4_SCSensor3_Task5.xlsx', 'idScene3_TempSensor3_Task4.xlsx', 'idScene6_SCSensor3_Task6.xlsx', 'idScene1_SCSensor2_Task1.xlsx', 'idScene1_TempSensor1_Task1.xlsx', 'idScene1_TempSensor4_Task1.xlsx', 'idScene5_SCSensor2_Task6.xlsx']
All task files found:  ['Task6HRSensor4.xlsx', 'Task1SCSenso

: 

In [ ]:
def get_sensor_type(filename):
    if "HR" in filename: return "HR"
    if "SC" in filename: return "SC"
    if "Temp" in filename: return "Temp"
    return None

: 

In [ ]:
new_scene_folder = os.path.join(source_path, "Scenes_Complete_Gina")
scene_files = os.listdir(new_scene_folder)
sonifications_path = os.path.join(target_path, "Sonifications")
if not os.path.exists(sonifications_path):
  os.makedirs(sonifications_path)

processed_scenes = []
for f in scene_files:
  scene_num = sceneNo(f)
  if scene_num not in processed_scenes:
    processed_scenes.append(scene_num)
  else:
    continue
  # If it doesn't exist already, create a folder for this scene
  scene_folder = os.path.join(sonifications_path, "Scene{}".format(scene_num))
  if not os.path.exists(scene_folder):
    os.makedirs(scene_folder)
  scene_files_filtered = fnmatch.filter(scene_files, "Scene{}*".format(scene_num))
  assert(len(scene_files_filtered) == 12)
  # Find all different sensor numbers in Scene_files_filtered. Sensors are like so in the file name: _SensorX_, where X = sensor_num
  sensor_nums = []
  for f in scene_files_filtered:
    sensor_num = f.split("_")[1].split("Sensor")[1]
    if sensor_num not in sensor_nums:
      sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    sensor_files_per_scene = fnmatch.filter(scene_files_filtered, "*Sensor{}*".format(sensor))
    assert(len(sensor_files_per_scene) == 3)
    # Create a new dataframe that concatenates these three files into a new one, removing the timestamp column and heading each new column with the appropriate bio code: "HR", "SC", "Temp"

    # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_scene_sorted = sorted(sensor_files_per_scene, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_scene_sorted:
      df = pd.read_excel(os.path.join(new_scene_folder, f))
      values = df.iloc[:, 1]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Scene {scene_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Scene{scene_num}AllSensor{sensor}.csv"
    final_path = os.path.join(scene_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Preview for Scene 4 Sensor 4
           HR        SC      Temp
0  202.105263  0.498923  32.65648
1  202.105263  0.498834  32.65648
2  202.105263  0.498923  32.65648
3  202.105263  0.499012  32.65648
4  202.105263  0.499278  32.65648
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4AllSensor4.csv
Preview for Scene 4 Sensor 1
          HR        SC       Temp
0  93.658537  1.477656  30.780236
1  93.658537  1.477656  30.780236
2  93.658537  1.477363  30.780236
3  93.658537  1.477363  30.780236
4  93.658537  1.477363  30.780236
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4AllSensor1.csv
Preview for Scene 4 Sensor 3
          HR        SC       Temp
0  93.249369  3.233553  31.109993
1  93.160857  3.234139  31.109993
2  93.072372  3.234432  31.109993
3  92.983917  3.235018  31.109993
4  92.895492  3.235018  31.109993
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Sonifications/Scene4/Scene4A

: 

In [ ]:
visualizations_path = os.path.join(target_path, 'Visualizations')
if not os.path.exists(visualizations_path):
  os.makedirs(visualizations_path)

task_files = os.listdir(tasks_path)

processed_tasks = []
for f in task_files:
  sensor_type = get_sensor_type(f)
  print('Sensor type: ', sensor_type)
  task_num = f.split(sensor_type)[0].split('Task')[1]
  print('Task number: ', task_num)
  if task_num not in processed_tasks:
    processed_tasks.append(task_num)
  else:
    continue

  # If it doesn't exist already, create a folder for this scene
  task_folder = os.path.join(visualizations_path, "Task{}".format(task_num))
  if not os.path.exists(task_folder):
    os.makedirs(task_folder)
  task_files_filtered = fnmatch.filter(task_files, "Task{}*".format(task_num))
  print(task_files_filtered)
  assert(len(task_files_filtered) == 12)
  # Find all different sensor numbers in task_files_filtered. Sensors are like so in the file name: *SensorX.xlsx, where X = sensor_num
  sensor_nums = []
  for f in task_files_filtered:
    print(f)
    sensor_num = f.split(".xlsx")[0].split("Sensor")[1]
    if sensor_num not in sensor_nums:
      sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    sensor_files_per_task = fnmatch.filter(task_files_filtered, "*Sensor{}*".format(sensor))
    assert(len(sensor_files_per_task) == 3)

    # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_task_sorted = sorted(sensor_files_per_task, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_task_sorted:
      df = pd.read_excel(os.path.join(tasks_path, f))
      values = df.iloc[:, 1]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Task {task_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)

    # Enforce column order HR -> SC -> Temp if present
    desired_cols = [c for c in ["HR", "SC", "Temp"] if c in final_df.columns]
    final_df = final_df[desired_cols]
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Task{task_num}AllSensor{sensor}.csv"
    final_path = os.path.join(task_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Sensor type:  HR
Task number:  6
['Task6HRSensor4.xlsx', 'Task6TempSensor2.xlsx', 'Task6SCSensor4.xlsx', 'Task6TempSensor4.xlsx', 'Task6TempSensor1.xlsx', 'Task6SCSensor3.xlsx', 'Task6HRSensor3.xlsx', 'Task6TempSensor3.xlsx', 'Task6SCSensor2.xlsx', 'Task6HRSensor2.xlsx', 'Task6HRSensor1.xlsx', 'Task6SCSensor1.xlsx']
Task6HRSensor4.xlsx
Task6TempSensor2.xlsx
Task6SCSensor4.xlsx
Task6TempSensor4.xlsx
Task6TempSensor1.xlsx
Task6SCSensor3.xlsx
Task6HRSensor3.xlsx
Task6TempSensor3.xlsx
Task6SCSensor2.xlsx
Task6HRSensor2.xlsx
Task6HRSensor1.xlsx
Task6SCSensor1.xlsx
Preview for Task 6 Sensor 4
          HR        SC       Temp
0  93.658537  0.603352  34.186254
1  93.658537  0.603352  34.186254
2  93.658537  0.603352  34.186254
3  93.658537  0.603175  34.186254
4  93.658537  0.602997  34.186254
Saved to /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task6/Task6AllSensor4.csv
Preview for Task 6 Sensor 2
          HR        SC       Temp
0  70.437510  2.009231  31.032687
1 

: 

Final thing I forgot; we should trim the files so that each column has the same number of lines!

In [ ]:
for folder in os.listdir(target_path):
  folder_path = os.path.join(target_path, folder)
  print('Folder: ', folder_path)
  for subfolder in os.listdir(folder_path):
    subfolder_path = os.path.join(folder_path, subfolder)
    print(f'Subfolder: {subfolder_path}')
    for f in os.listdir(subfolder_path):
      # Here we are either accessing scene or task files
      file_path = os.path.join(subfolder_path, f)
      print(f'Final path: {file_path}')
      # Read each csv
      df = pd.read_csv(file_path)
      # Find the length of each column
      lengths = [len(df[col]) for col in df.columns]
      # Find the minimum length
      min_length = min(lengths)
      # Trim the other two columns to have the same no. of lines
      for col in df.columns:
        if len(df[col]) != min_length:
          df = df.drop(df.tail(len(df[col]) - min_length).index)
      # print end to ensure it's correct
      print(df.tail())
      # Write output to same file
      df.to_csv(file_path, index=False)


Folder:  /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations
Subfolder: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor3.csv
               HR        SC       Temp
21089  112.941176  3.145934  31.016464
21090  112.941176  3.146227  31.016464
21091  112.941176  3.146813  31.016464
21092  112.941176  3.146813  31.016464
21093  112.941176  3.146813  31.016464
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor4.csv
               HR        SC       Temp
21113  182.857143  2.461099  32.582459
21114  182.857143  2.460806  32.582459
21115  182.857143  2.460806  32.582459
21116  182.857143  2.460806  32.582459
21117  182.857143  2.460806  32.582459
Final path: /home/gina101/new_data/Ljubljana/Environment/Session2/Visualizations/Task2/Task2AllSensor2.csv
              HR        SC       Temp
21092  73

: 

## Modification for Rennes

In [29]:
def findSensorNoInTaskFiles(name):
  sensor_type = get_sensor_type(name)
  print('Sensor type in findSensorNoInTaskFiles: ', sensor_type)
  sensor_num = name.split(".xlsx")[0].split(sensor_type)[1]
  print('sensor_num: ', sensor_num)
  return sensor_num

def sceneNo(name):
  sensor_type = get_sensor_type(name)
  return name.split(sensor_type)[0].split("Scene")[1]

In [30]:
def findTaskNo(name):
  return name.split("Task")[1].split('.xlsx')[0]

In [34]:
# Find Scene ID files
all_files = os.listdir(source_path)
id_files = fnmatch.filter(all_files, "idScene*.xlsx")
scene_files = fnmatch.filter(all_files, "Scene*.xlsx")
task_files = fnmatch.filter(all_files, "Task*.xlsx")

# Print these files
print("All idScene files found: ", id_files)
print("All task files found: ", task_files)

processed_scenes = []

def get_sensor_type(filename):
    if "HR" in filename: return "HR"
    if "SC" in filename: return "SC"
    if "Temp" in filename: return "Temp"
    return None

for id_scene_file in id_files:
  all_filtered_task_files = []
  # Find Scene No and Task No from the id_scene_file
  scene_num = sceneNo(id_scene_file)
  if scene_num in processed_scenes:
    continue
  processed_scenes.append(scene_num)

  task_num = findTaskNo(id_scene_file)
  print(f"Processing idScene file: {id_scene_file} (Scene no: {scene_num}, Task no: {task_num})")

  task_files_for_this_scene = fnmatch.filter(task_files, "Task{}*".format(task_num))
  # Keep only the desired sensor/participant combos
  wanted_suffixes = ("HR2.xlsx", "HR5.xlsx", "Temp2.xlsx", "Temp5.xlsx", "SC2.xlsx", "SC5.xlsx", "HR3.xlsx", "Temp3.xlsx", "SC3.xlsx", "HR6.xlsx", "Temp6.xlsx", "SC6.xlsx")
  task_files_for_this_scene = [f for f in task_files_for_this_scene if any(f.endswith(suf) for suf in wanted_suffixes)]
  print(f"Filtered task files for Scene {scene_num}, Task {task_num}: {task_files_for_this_scene}")
  assert(len(task_files_for_this_scene) == 12)

  # Open id_scene_file and read cells A1 and A2
  id_scene_file_path = os.path.join(source_path, id_scene_file)
  df_id_scene = pd.read_excel(id_scene_file_path, header=None)
  # print(df_id_scene)
  starting_timestamp = df_id_scene.iloc[0, 0].astype(int)
  ending_timestamp = df_id_scene.iloc[1, 0].astype(int)
  print(f"Start timestamp: {starting_timestamp}, End timestamp: {ending_timestamp}")

  # Now, time to find the appropriate lines to 'splice' the files
  # Read through each one of the task files and find the lines that have the values of cell_a1 and cell_a2

  for task_file in task_files_for_this_scene:
      task_file_path = os.path.join(source_path, task_file)
      df_task = pd.read_excel(task_file_path)

      # Convert the first column of df_task to integer type for comparison
      # Use a tolerance to find the nearest timestamp when exact matches don't exist
      col = df_task.iloc[:, 1].astype(float)
      tol = 10  # adjust this tolerance as needed (same units as timestamps)
      
      # find nearest index for start timestamp
      start_diff = (col - float(starting_timestamp)).abs()
      start_idx = start_diff.idxmin()
      if start_diff.loc[start_idx] <= tol:
        start_index_series = pd.Index([start_idx])
        print(f"Start index found at {start_idx} (value={col.loc[start_idx]}, diff={start_diff.loc[start_idx]}) for {task_file}")
      else:
        start_index_series = pd.Index([])
        print(f"Start index not found within tolerance {tol} for {task_file}, nearest diff={start_diff.loc[start_idx]}")
      
      # find nearest index for end timestamp
      end_diff = (col - float(ending_timestamp)).abs()
      end_idx = end_diff.idxmin()
      if end_diff.loc[end_idx] <= tol:
        end_index_series = pd.Index([end_idx])
        print(f"End index found at {end_idx} (value={col.loc[end_idx]}, diff={end_diff.loc[end_idx]}) for {task_file}")
      else:
        end_index_series = pd.Index([])
        print(f"End index not found within tolerance {tol} for {task_file}, nearest diff={end_diff.loc[end_idx]}")

      if not start_index_series.empty and not end_index_series.empty:
          start_row = start_index_series[0]
          end_row = end_index_series[0]
          if start_row <= end_row:
            # Create new dataframe and copy the rows
            spliced_df = df_task.iloc[start_row : end_row + 1]

            # Extract sensor type and number for column naming
            sensor_type = get_sensor_type(task_file)
            sensor_num = findSensorNoInTaskFiles(task_file)
            print(f"Task file {task_file} has sensor type {sensor_type} and belongs to participant {sensor_num}")

            if sensor_type and sensor_num:
              # Assuming the first column is 'Sample_no', the second is 'Timestamp', and the third is the value
              spliced_df.columns = ['Sample_no', 'Timestamp', 'Value']

          else:
            print(f"Warning: Start index ({start_row}) is greater than end index ({end_row}) for {task_file}. Skipping splicing.")

          output_filename = f"Scene{scene_num}_{sensor_type}Sensor{sensor_num}_Task{task_num}.xlsx"
          output_path = os.path.join(target_path, output_filename)
          
          dirpath = os.path.dirname(output_path)
          if dirpath:
            os.makedirs(dirpath, exist_ok=True)
          spliced_df.to_excel(output_path, index=False)
          print(f"Saved spliced data for {task_file} to {output_path}")
      else:
        print(f"No data merged for Scene {scene_num}, Task {task_num}")

All idScene files found:  ['idScene3HR2-Task7.xlsx', 'idScene2Temp1-Task7.xlsx', 'idScene5Temp6-Task8.xlsx', 'idScene2SC1-Task7.xlsx', 'idScene3SC3-Task7.xlsx', 'idScene3SC2-Task7.xlsx', 'idScene1SC3-Task6.xlsx', 'idScene4SC2-Task8.xlsx', 'idScene6SC5-Task8.xlsx', 'idScene2Temp2-Task7.xlsx', 'idScene5Temp5-Task8.xlsx']
All task files found:  ['Task8HR3.xlsx', 'Task7Temp3.xlsx', 'Task6Temp2.xlsx', 'Task7Temp2.xlsx', 'Task6Temp3.xlsx', 'Task8SC6.xlsx', 'Task6Temp6.xlsx', 'Task7SC5.xlsx', 'Task7SC3.xlsx', 'Task6SC5.xlsx', 'Task7HR6.xlsx', 'Task7Temp5.xlsx', 'Task6HR6.xlsx', 'Task7HR3.xlsx', 'Task8Temp3.xlsx', 'Task7Temp1.xlsx', 'Task7SC2.xlsx', 'Task6SC2.xlsx', 'Task8SC5.xlsx', 'Task6SC6.xlsx', 'Task6HR3.xlsx', 'Task8SC2.xlsx', 'Task6SC3.xlsx', 'Task7SC1.xlsx', 'Task7HR5.xlsx', 'Task7Temp6.xlsx', 'Task6Temp5.xlsx', 'Task8SC1.xlsx', 'Task6Temp1.xlsx', 'Task8HR5.xlsx', 'Task6HR5.xlsx', 'Task8Temp2.xlsx', 'Task8Temp6.xlsx', 'Task8Temp1.xlsx', 'Task7HR2.xlsx', 'Task6HR2.xlsx', 'Task7SC6.xlsx'

In [37]:
new_scene_folder = os.path.join("/home/gina101/Rennes/session2/Scenes_Complete_Gina")
scene_files = os.listdir(new_scene_folder)
sonifications_path = os.path.join(target_path, "Sonifications")
if not os.path.exists(sonifications_path):
  os.makedirs(sonifications_path)

processed_scenes = []
for f in scene_files:
  scene_num = sceneNo(f)
  if scene_num not in processed_scenes:
    processed_scenes.append(scene_num)
  else:
    continue
  # If it doesn't exist already, create a folder for this scene
  scene_folder = os.path.join(sonifications_path, "Scene{}".format(scene_num))
  if not os.path.exists(scene_folder):
    os.makedirs(scene_folder)
  scene_files_filtered = fnmatch.filter(scene_files, "Scene{}*".format(scene_num))
  assert(len(scene_files_filtered) == 12)
  # Find all different sensor numbers in Scene_files_filtered. Sensors are like so in the file name: _SensorX_, where X = sensor_num
  sensor_nums = []
  for f in scene_files_filtered:
    sensor_num = f.split("_")[1].split("Sensor")[1]
    if sensor_num not in sensor_nums:
      sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    sensor_files_per_scene = fnmatch.filter(scene_files_filtered, "*Sensor{}*".format(sensor))
    assert(len(sensor_files_per_scene) == 3)
    # Create a new dataframe that concatenates these three files into a new one, removing the timestamp column and heading each new column with the appropriate bio code: "HR", "SC", "Temp"

    # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_scene_sorted = sorted(sensor_files_per_scene, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_scene_sorted:
      df = pd.read_excel(os.path.join(new_scene_folder, f))
      values = df.iloc[:, 2]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Scene {scene_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Scene{scene_num}AllSensor{sensor}.csv"
    final_path = os.path.join(scene_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Preview for Scene 5_ Sensor 5
           HR         SC       Temp
0  142.222222  14.690304  31.126733
1  142.222222  14.695409  31.126733
2  142.222222  14.685199  31.126733
3  142.222222  14.685199  31.126733
4  142.222222  14.685199  31.126733
Saved to /home/gina101/new_data/Rennes/Environment/Session2/Sonifications/Scene5_/Scene5_AllSensor5.csv
Preview for Scene 5_ Sensor 6
           HR        SC       Temp
0  156.734694  6.971194  32.279969
1  156.734694  6.976299  32.279969
2  156.734694  6.976299  32.279969
3  156.734694  6.971194  32.279969
4  156.734694  6.976299  32.279969
Saved to /home/gina101/new_data/Rennes/Environment/Session2/Sonifications/Scene5_/Scene5_AllSensor6.csv
Preview for Scene 5_ Sensor 2
      HR        SC       Temp
0  128.0  8.890761  32.294543
1  128.0  8.890761  32.294543
2  128.0  8.890761  32.294543
3  128.0  8.885656  32.294543
4  128.0  8.885656  32.294543
Saved to /home/gina101/new_data/Rennes/Environment/Session2/Sonifications/Scene5_/Scene5_AllSens

In [40]:
visualizations_path = os.path.join(target_path, 'Visualizations')
if not os.path.exists(visualizations_path):
  os.makedirs(visualizations_path)

task_files = fnmatch.filter(os.listdir(source_path), "Task*.xlsx")
# Filter only for sensors 2 and 5, as these are the only valid ones
# wanted_suffixes = ("HR2.xlsx", "HR5.xlsx", "Temp2.xlsx", "Temp5.xlsx", "SC2.xlsx", "SC5.xlsx")
# task_files = [f for f in task_files if any(f.endswith(suf) for suf in wanted_suffixes)]
# print("Filtered task files: ", task_files)
# assert(len(task_files) == 18)

processed_tasks = []
for f in task_files:
  sensor_type = get_sensor_type(f)
  print('Sensor type: ', sensor_type)
  task_num = f.split(sensor_type)[0].split('Task')[1]
  print('Task number: ', task_num)
  if task_num not in processed_tasks:
    processed_tasks.append(task_num)
  else:
    continue

  # If it doesn't exist already, create a folder for this task
  task_folder = os.path.join(visualizations_path, "Task{}".format(task_num))
  if not os.path.exists(task_folder):
    os.makedirs(task_folder)
  task_files_filtered = fnmatch.filter(task_files, "Task{}*".format(task_num))
  wanted_suffixes = ("HR2.xlsx", "HR5.xlsx", "Temp2.xlsx", "Temp5.xlsx", "SC2.xlsx", "SC5.xlsx", "HR3.xlsx", "Temp3.xlsx", "SC3.xlsx", "HR6.xlsx", "Temp6.xlsx", "SC6.xlsx")
  task_files_filtered = [f for f in task_files_filtered if any(f.endswith(suf) for suf in wanted_suffixes)]
  print(task_files_filtered)
  assert(len(task_files_filtered) == 12)
  # Find all different sensor numbers in task_files_filtered. Sensors are like so in the file name: *SensorX.xlsx, where X = sensor_num
  sensor_nums = [2, 3, 5, 6]
  # for f in task_files_filtered:
  #   print(f)
  #   sensor_num = f.split(".xlsx")[0].split("Sensor")[1]
  #   if sensor_num not in sensor_nums:
  #     sensor_nums.append(sensor_num)
  for sensor in sensor_nums:
    # Only keep files that match this pattern TaskXHRY.xlsx, TaskXSCY.xlsx, TaskXTempY.xlsx where Y = sensor_num
    sensor_files_per_task = fnmatch.filter(task_files_filtered, "Task{}*{}*".format(task_num, sensor))
    print('sensor_files_per_task: ', sensor_files_per_task)
    assert(len(sensor_files_per_task) == 3)

  # Ensure the files are concatenated in HR -> SC -> Temp order
    order = {"HR": 0, "SC": 1, "Temp": 2}
    sensor_files_per_task_sorted = sorted(sensor_files_per_task, key=lambda x: order.get(get_sensor_type(x), 99))

    dfs = []
    for f in sensor_files_per_task_sorted:
      df = pd.read_excel(os.path.join(source_path, f))
      values = df.iloc[:, 2]
      df_clean = pd.DataFrame({get_sensor_type(f): values})
      dfs.append(df_clean)

    print(f"Preview for Task {task_num} Sensor {sensor}")
    # Concatenate into a final dataframe horizontally
    final_df = pd.concat(dfs, axis=1)

    # Enforce column order HR -> SC -> Temp if present
    desired_cols = [c for c in ["HR", "SC", "Temp"] if c in final_df.columns]
    final_df = final_df[desired_cols]
    # Print first few rows to ensure correctness
    print(final_df.head())

    # Save to output file
    out_name = f"Task{task_num}AllSensor{sensor}.csv"
    final_path = os.path.join(task_folder, out_name)
    final_df.to_csv(final_path, index=False)
    print(f"Saved to {final_path}")


Sensor type:  HR
Task number:  8
['Task8HR3.xlsx', 'Task8SC6.xlsx', 'Task8Temp3.xlsx', 'Task8SC5.xlsx', 'Task8SC2.xlsx', 'Task8HR5.xlsx', 'Task8Temp2.xlsx', 'Task8Temp6.xlsx', 'Task8HR2.xlsx', 'Task8Temp5.xlsx', 'Task8HR6.xlsx', 'Task8SC3.xlsx']
sensor_files_per_task:  ['Task8SC2.xlsx', 'Task8Temp2.xlsx', 'Task8HR2.xlsx']
Preview for Task 8 Sensor 2
          HR        SC       Temp
0  63.471074  6.802721  32.269897
1  63.471074  6.802721  32.269897
2  63.471074  6.802721  32.269897
3  63.471074  6.797616  32.269897
4  63.471074  6.797616  32.269897
Saved to /home/gina101/new_data/Rennes/Environment/Session2/Visualizations/Task8/Task8AllSensor2.csv
sensor_files_per_task:  ['Task8HR3.xlsx', 'Task8Temp3.xlsx', 'Task8SC3.xlsx']
Preview for Task 8 Sensor 3
           HR        SC       Temp
0  170.666667  1.625641  30.988595
1  170.666667  1.625641  30.988595
2  170.666667  1.625641  30.988595
3  170.666667  1.625641  30.988595
4  170.666667  1.625641  30.988595
Saved to /home/gina101/new_

In [41]:
for folder in os.listdir(target_path):
  folder_path = os.path.join(target_path, folder)
  print('Folder: ', folder_path)
  for subfolder in os.listdir(folder_path):
    subfolder_path = os.path.join(folder_path, subfolder)
    print(f'Subfolder: {subfolder_path}')
    for f in os.listdir(subfolder_path):
      # Here we are either accessing scene or task files
      file_path = os.path.join(subfolder_path, f)
      print(f'Final path: {file_path}')
      # Read each csv
      df = pd.read_csv(file_path)
      # Find the length of each column
      lengths = [len(df[col]) for col in df.columns]
      # Find the minimum length
      min_length = min(lengths)
      # Trim the other two columns to have the same no. of lines
      for col in df.columns:
        if len(df[col]) != min_length:
          df = df.drop(df.tail(len(df[col]) - min_length).index)
      # print end to ensure it's correct
      print(df.tail())
      # Write output to same file
      df.to_csv(file_path, index=False)


Folder:  /home/gina101/new_data/Rennes/Environment/Session2/Visualizations
Subfolder: /home/gina101/new_data/Rennes/Environment/Session2/Visualizations/Task6
Final path: /home/gina101/new_data/Rennes/Environment/Session2/Visualizations/Task6/Task6AllSensor5.csv
                HR         SC       Temp
135198  147.692308  13.235313  30.899741
135199  147.692308  13.235313  30.899741
135200  147.692308  13.235313  30.899741
135201  147.692308  13.235313  30.899741
135202  147.692308  13.235313  30.899741
Final path: /home/gina101/new_data/Rennes/Environment/Session2/Visualizations/Task6/Task6AllSensor6.csv
          HR        SC       Temp
135198  96.0  7.997345  30.918158
135199  96.0  8.002451  30.918158
135200  96.0  8.012661  30.918158
135201  96.0  8.012661  30.918158
135202  96.0  8.012661  30.918158
Final path: /home/gina101/new_data/Rennes/Environment/Session2/Visualizations/Task6/Task6AllSensor3.csv
               HR        SC       Temp
135197  67.964602  1.974359  30.427862
13

## From this point forward, this code is usable for Eleusina

In [12]:
def findSection(name):
  return name.split("-")[1].split('.xlsx')[0]

In [13]:
def getSensorType(name):
  if "HR" in name:
    return "HR"
  elif "SC" in name:
    return "SC"
  elif "Temp" in name:
    return "Temp"
  return None

In [14]:
def sceneNo(name):
  sensor_type = getSensorType(name)
  return name.split(sensor_type)[0].split("Scene")[1]

In [15]:
def findTaskNo(name):
  return name.split("Task")[1].split(getSensorType(name))[0]

In [16]:
# current_folder = os.path.join(source_path, 'Tasks')
current_folder = tasks_path
file_list = [n for n in os.listdir(current_folder) if not fnmatch.fnmatch(n, "time*")]

In [17]:
sections = {}
for f in file_list:
  section = findSection(f)
  task = findTaskNo(f)
  if section not in sections:
    sections[section] = {task: [f]}
  else:
    if task in sections[section]:
      sections[section][task].append(f)
    else:
      sections[section][task] = [f]

In [18]:
sections

{'Introduction': {'5': ['Task5HRSensor2-Introduction.xlsx',
   'Task5SCSensor4-Introduction.xlsx',
   'Task5TempSensor4-Introduction.xlsx',
   'Task5SCSensor2-Introduction.xlsx',
   'Task5SCSensor5-Introduction.xlsx',
   'Task5HRSensor4-Introduction.xlsx',
   'Task5HRSensor3-Introduction.xlsx',
   'Task5TempSensor5-Introduction.xlsx',
   'Task5TempSensor3-Introduction.xlsx',
   'Task5SCSensor3-Introduction.xlsx',
   'Task5SCSensor6-Introduction.xlsx',
   'Task5TempSensor6-Introduction.xlsx',
   'Task5TempSensor2-Introduction.xlsx',
   'Task5HRSensor6-Introduction.xlsx',
   'Task5HRSensor5-Introduction.xlsx'],
  '4': ['Task4SCSensor5-Introduction.xlsx',
   'Task4HRSensor4-Introduction.xlsx',
   'Task4HRSensor6-Introduction.xlsx',
   'Task4SCSensor4-Introduction.xlsx',
   'Task4TempSensor3-Introduction.xlsx',
   'Task4SCSensor6-Introduction.xlsx',
   'Task4HRSensor2-Introduction.xlsx',
   'Task4SCSensor3-Introduction.xlsx',
   'Task4TempSensor6-Introduction.xlsx',
   'Task4TempSensor4-In

In [19]:
def findSensorNo(name):
    return name.split('-')[0].split('Sensor')[1]

In [ ]:
for section in sections.keys():
  for task in sections[section].keys():
    sensors = []
    for f in sections[section][task]:
      sensor = findSensorNo(f)
      if sensor not in sensors:
        sensors.append(sensor)
    for sensor in sensors:
      sensor_files = fnmatch.filter(sections[section][task], '*Sensor{}*'.format(sensor))
      [hr_file] = fnmatch.filter(sensor_files, "*HR*")
      [sc_file] = fnmatch.filter(sensor_files, "*SC*")
      [temp_file] = fnmatch.filter(sensor_files, "*Temp*")
      hr_pd = pd.read_excel(os.path.join(current_folder, hr_file))
      sc_pd = pd.read_excel(os.path.join(current_folder, sc_file))
      temp_pd = pd.read_excel(os.path.join(current_folder, temp_file))
      res_pd = pd.concat([hr_pd, sc_pd, temp_pd], axis=1)
      new_file_name = 'Task{}AllSensor{}-{}.csv'.format(task, sensor, section)
      new_path = os.path.join(target_path, new_file_name)
      res_pd.to_csv(new_path, index=False, header=['HR', 'SC', 'Temp'])

### Making the complete Scene files before we make sonification files

In [22]:
def findSectionAndTask(name):
    section = name.split('-')[1]
    number = name.split('-')[2].split('.xlsx')[0]
    return section, number

In [23]:
# Find Scene ID files
id_files = os.listdir(ids_path)
scene_files = os.listdir(scenes_path)
task_files = os.listdir(tasks_path)

# Print these files
print("All idScene files found: ", id_files)
print("All task files found: ", task_files)

processed_scenes = []

for id_scene_file in id_files:
  all_filtered_task_files = []
  # Find Scene No and Task No from the id_scene_file
  scene_num = sceneNo(id_scene_file)
  if scene_num in processed_scenes:
    continue
  processed_scenes.append(scene_num)

  [section, task_num] = findSectionAndTask(id_scene_file)
  print(f"Processing idScene file: {id_scene_file} (Scene no: {scene_num}, Section: {section}, Task no: {task_num})")

  task_files_for_this_scene = fnmatch.filter(task_files, "Task{}*-{}.xlsx".format(task_num, section))
  # print(f"Task files for this scene: {task_files_for_this_scene}")
  assert(len(task_files_for_this_scene) == 15)

  # # Open id_scene_file and read cells A1 and A2
  id_scene_file_path = os.path.join(ids_path, id_scene_file)
  df_id_scene = pd.read_excel(id_scene_file_path, header=None)
  # print(df_id_scene)
  starting_line = df_id_scene.iloc[0, 0].astype(int)
  ending_line = df_id_scene.iloc[1, 0].astype(int)
  print(f"Start line: {starting_line}, End line: {ending_line}")

  # Now, time to find the appropriate lines to 'splice' the files
  # Read through each one of the task files and find the lines that have the values of cell_a1 and cell_a2

  for task_file in task_files_for_this_scene:
    task_file_path = os.path.join(tasks_path, task_file)
    df_task = pd.read_excel(task_file_path)

    # Convert the first column of df_task to integer type for comparison
    new_df = df_task.iloc[starting_line : ending_line]
    # print(new_df)

    # Extract sensor type and number for column naming
    sensor_type = getSensorType(task_file)
    sensor_num = findSensorNo(task_file)
    print(f"Task file {task_file} has sensor type {sensor_type} and belongs to participant {sensor_num}")

    if sensor_type and sensor_num:
      output_filename = f"Scene{scene_num}{sensor_type}Sensor{sensor_num}-{section}-{task_num}.xlsx"
      scenes_output_path = os.path.join(source_path, 'Scenes_Complete_Gina')
      if not os.path.exists(scenes_output_path):
        os.makedirs(scenes_output_path)
      output_path = os.path.join(scenes_output_path, output_filename)
      new_df.to_excel(output_path, index=False, header=None)
      print(f"Saved spliced data for {task_file} to {output_path}")
    else:
      print(f"No data merged for Scene {scene_num}, Task {task_num}")

All idScene files found:  ['idScene123HRSensor3-Main-5.xlsx', 'idScene76SCSensor4-Main-3.xlsx', 'idScene50SCSensor4-Main-2.xlsx', 'idScene41TempSensor5-Main-1.xlsx', 'idScene50TempSensor4-Main-2.xlsx', 'idScene76TempSensor5-Main-4.xlsx', 'idScene76HRSensor6-Main-3.xlsx', 'idScene81TempSensor6-Main-4.xlsx', 'idScene25SCSensor5-Introduction-5.xlsx', 'idScene81HRSensor2-Main-4.xlsx', 'idScene50TempSensor6-Main-2.xlsx', 'idScene81SCSensor4-Main-4.xlsx', 'idScene81SCSensor2-Main-4.xlsx', 'idScene50SCSensor5-Main-2.xlsx', 'idScene119HRSensor5-Main-5.xlsx', 'idScene76SCSensor5-Main-3.xlsx', 'idScene76HRSensor4-Main-4.xlsx', 'idScene15HRSensor6-Introduction-4.xlsx', 'idScene50HRSensor5-Main-2.xlsx', 'idScene25HRSensor3-Introduction-5.xlsx', 'idScene81HRSensor6-Main-4.xlsx', 'idScene117HRSensor2-Main-5.xlsx', 'idScene50HRSensor4-Main-2.xlsx', 'idScene76TempSensor3-Main-3.xlsx', 'idScene54SCSensor6-Main-2.xlsx', 'idScene81SCSensor6-Main-4.xlsx', 'idScene81HRSensor5-Main-4.xlsx', 'idScene161TempS

In [38]:
# Now, we need to merge these scene files into final files per scene and sensor
new_scenes_path = os.path.join(source_path, 'Scenes_Complete_Gina')
print('Scenes output path: ', new_scenes_path)
scene_files = os.listdir(new_scenes_path)
processed_scenes = []
sensor_nums = ['4']
for f in scene_files:
    scene_num = sceneNo(f)
    if scene_num in processed_scenes:
        continue
    processed_scenes.append(scene_num)
    # find all files for this scene
    scene_files_for_this_scene = fnmatch.filter(scene_files, "Scene{}*".format(scene_num))
    # print(f"Scene files for Scene {scene_num}: {scene_files_for_this_scene}")
    assert(len(scene_files_for_this_scene) == 15)
    # Find all files for each sensor number
    for sensor in sensor_nums:
        sensor_files = fnmatch.filter(scene_files_for_this_scene, "*Sensor{}*".format(sensor))
        # print(f"Sensor files for Scene {scene_num} Sensor {sensor}: {sensor_files}")
        assert(len(sensor_files) == 3)
        dfs = []
        for f in sensor_files:
            path = os.path.join(new_scenes_path, f)
            print(f'About to read file: {f} for Scene {scene_num} Sensor {sensor}')
            df = pd.read_excel(path, header=None)
            values = df.iloc[:, 0]
            df_clean = pd.DataFrame({getSensorType(f): values})
            dfs.append(df_clean)
    
        print(f"Preview for Scene {scene_num} Sensor {sensor}")
        # Concatenate into a final dataframe horizontally
        new_df = pd.concat(dfs, axis=1)
    
        # Enforce column order HR -> SC -> Temp if present
        desired_cols = [c for c in ["HR", "SC", "Temp"] if c in new_df.columns]
        new_df = new_df[desired_cols]
        # Print first few rows to ensure correctness
        print(new_df.head())
    
        # Save to output file
        output_filename = f"Scene{scene_num}AllSensor{sensor}.csv"
        scenes_output_path = os.path.join(target_path, 'Sonifications', f'Scene{scene_num}')
        if not os.path.exists(scenes_output_path):
            os.makedirs(scenes_output_path)
        new_df.to_csv(os.path.join(scenes_output_path, output_filename), index=False, header=['HR', 'SC', 'Temp'])

Scenes output path:  /home/gina101/Eleusis 2.0/Eleusis 2.0/Scenes_Complete_Gina
About to read file: Scene119TempSensor4-Main-5.xlsx for Scene 119 Sensor 4
About to read file: Scene119SCSensor4-Main-5.xlsx for Scene 119 Sensor 4
About to read file: Scene119HRSensor4-Main-5.xlsx for Scene 119 Sensor 4
Preview for Scene 119 Sensor 4
          HR        SC       Temp
0  73.846154  0.985459  30.135091
1  73.846154  0.985903  30.145015
2  73.846154  0.985903  30.145015
3  73.846154  0.985903  30.145015
4  73.846154  0.986347  30.145015
About to read file: Scene36HRSensor4-Main-1.xlsx for Scene 36 Sensor 4
About to read file: Scene36SCSensor4-Main-1.xlsx for Scene 36 Sensor 4
About to read file: Scene36TempSensor4-Main-1.xlsx for Scene 36 Sensor 4
Preview for Scene 36 Sensor 4
          HR        SC     Temp
0  73.846154  3.317949  30.4229
1  73.846154  3.320879  30.4229
2  73.846154  3.322344  30.4229
3  73.846154  3.325275  30.4229
4  73.846154  3.326740  30.4229
About to read file: Scene12

### Let's make sure that there aren't any variances per line inside each file

In [39]:
sonifications_path = os.path.join(target_path, "Sonifications")
visualizations_path = os.path.join(target_path, 'Visualizations')
# for folder in os.listdir(visualizations_path):
#   section_path = os.path.join(visualizations_path, folder)
#   for section_folder in os.listdir(section_path):
#     task_path = os.path.join(section_path, section_folder)
#     for task_folder in os.listdir(task_path):
#       file_path = os.path.join(task_path, task_folder)
#       df = pd.read_csv(file_path)
#       # Find the length of each column
#       lengths = [len(df[col]) for col in df.columns]
#       # Find the minimum length
#       min_length = min(lengths)
#       # Trim the other two columns to have the same no. of lines
#       for col in df.columns:
#         if len(df[col]) != min_length:
#           df = df.drop(df.tail(len(df[col]) - min_length).index)
#       # print end to ensure it's correct
#       print(df.tail())
#       # Write output to same file
#       df.to_csv(file_path, index=False)

for folder in os.listdir(sonifications_path):
  print('Folder: ', folder_path)
  folder_path = os.path.join(sonifications_path, folder)
  for f in os.listdir(folder_path):
    # Here we are either accessing scene or task files
    file_path = os.path.join(folder_path, f)
    print(f'Final path: {file_path}')
    # Read each csv
    df = pd.read_csv(file_path)
    # Find the length of each column
    lengths = [len(df[col]) for col in df.columns]
    # Find the minimum length
    min_length = min(lengths)
    # Trim the other two columns to have the same no. of lines
    for col in df.columns:
      if len(df[col]) != min_length:
        df = df.drop(df.tail(len(df[col]) - min_length).index)
    # print end to ensure it's correct
    print(df.tail())
    # Write output to same file
    df.to_csv(file_path, index=False)


Folder:  /home/gina101/new_data/Eleusis 2.0/Environment/Session1/Sonifications/Scene161
Final path: /home/gina101/new_data/Eleusis 2.0/Environment/Session1/Sonifications/Scene161/Scene161AllSensor4.csv
            HR        SC       Temp
634  64.322689  0.619603  31.881807
635  64.430252  0.619603  31.881807
636  64.537815  0.619603  31.881807
637  64.537815  0.619603  31.881807
638  64.537815  0.619603  31.881807
Final path: /home/gina101/new_data/Eleusis 2.0/Environment/Session1/Sonifications/Scene161/Scene161AllSensor5.csv
            HR        SC       Temp
634  76.039604  1.208791  30.351572
635  76.039604  1.208791  30.351572
636  76.039604  1.209235  30.351572
637  76.039604  1.209235  30.351572
638  76.039604  1.209235  30.351572
Final path: /home/gina101/new_data/Eleusis 2.0/Environment/Session1/Sonifications/Scene161/Scene161AllSensor2.csv
             HR        SC       Temp
634  130.169492  2.450549  32.246478
635  130.169492  2.446154  32.246478
636  130.169492  2.446154  

# Trimming the ends of tasks that have a scene at the end that continues to the next task

In [ ]:
session_task_files_path = r'/content/drive/MyDrive/TT8/new_data/Thematic3/Session2/Visualizations'

Detection mechanism

In [ ]:
# Split scenes:
scenes = [21, 105]

In [ ]:
# First task in each scene
first_tasks = {
    21: ['Section 2', 2],
    105: ['Section 3', 3]
}

Find the ending line

In [ ]:
ending_rows = {
    21: 134752,
    105: 29413
}

Trim the files with different line counts accordingly.

1. Find files of this task in visualization folder
2. Find files that have lines greater than the one in ending_rows.
3. Remove lines
4. Save
5. Redownload



In [ ]:
for scene in first_tasks.keys():
  section_folder = os.path.join(session_task_files_path, first_tasks[scene][0])
  task_folder = os.path.join(section_folder, 'Task{}'.format(first_tasks[scene][1]))
  files = os.listdir(task_folder)
  for f in files:
    file_path = os.path.join(task_folder, f)
    df = pd.read_csv(file_path, header=0)
    diff = df.shape[0] - ending_rows[scene]
    if df.shape[0] > ending_rows[scene]:
      print('LARGE')
      print(f)
      df.drop(df.tail(diff).index, inplace = True)
      df.to_csv(file_path, index=False)
      print(df.shape[0])
    else:
      print('BASELINE')
      print(df.shape[0])

BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
134752
BASELINE
29413
BASELINE
29413
BASELINE
29413
BASELINE
29413
BASELINE
29394
